In [ ]:
import torch
import os

# --- HARDWARE ACCELERATION ---
num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device} | CPU Cores activated: {num_cores}")

Esecuzione su: cpu | Core CPU attivati: 6


In [ ]:
import numpy as np
import mne
from moabb.datasets import PhysionetMI
from moabb.paradigms import MotorImagery
import gc
import pandas as pd
from EEGNet import EEGNet

# 1. Initialize the PhysioNet Motor Imagery dataset
dataset = PhysionetMI()

# 2. Motor Imagery paradigm setup
# Resampling to 160Hz (exactly as required by EEGNet)
# Typical bandpass filter for MI: 8 - 32 Hz (sensorimotor rhythms)
paradigm = MotorImagery(
    resample=160,
    fmin=8.0,
    fmax=32.0,
    events=['left_hand', 'right_hand'], # Starting with a classic binary task
    n_classes=2,
    tmin=1.0,
    tmax=3.0,
)

# You can add more subjects here without crashing
subject_list = list(range(1, 16)) 

# --- 3. INCREMENTAL LOADING (RAM-FRIENDLY) ---
X_list, y_list, meta_list = [], [], []

print(f"\nStarting incremental loading for {len(subject_list)} subjects...")
for sbj in subject_list:
    print(f"Extracting Subject {sbj}...")
    # Single subject extraction
    X_tmp, y_tmp, meta_tmp = paradigm.get_data(dataset=dataset, subjects=[sbj])
    
    # Immediate conversion to float32 and storage
    X_list.append(X_tmp.astype('float32'))
    y_list.append(y_tmp)
    meta_list.append(meta_tmp)
    
    # RAM Cleanup
    del X_tmp
    gc.collect()

# Final concatenation
X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\nLoading completed!")
print(f"Final shape (X): {X.shape} -> (trials, channels, samples)")

```text
Final shape (X): (675, 64, 321) -> (trials, channels, samples)

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time
from utils import calculate_advanced_metrics, plot_confusion_matrix

# 1. Label Preparation (0 for left_hand, 1 for right_hand)
labels_dict = {val: i for i, val in enumerate(np.unique(y))}
y_int = np.array([labels_dict[label] for label in y], dtype=np.int64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y_int).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

# Training Parameters
num_epochs = 50 # EEGNet is very lightweight, we can run more epochs
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
loso_results = []

subjects_array = metadata['subject'].unique()
print(f"\n=== Starting LOSO Validation on PhysioNet MI ({len(subjects_array)} Subjects) ===")

all_y_true_latent = []
all_y_pred_latent = []

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    # Masks for Training (N-1) and Test (1)
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    # Reinitialize the model from scratch for each LOSO iteration
    model = EEGNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    # --- TRAINING LOOP ---
    model.train()
    start_time = time.time()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        
        # Iterate one subject at a time (Batch = 1 full subject)
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask].to(device)
            batch_y = y_tensor[subj_mask].to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Pass the exact number of trials for THIS subject
            # Latent alignment will calculate perfect mean/variance for it
            current_sbj_trials = batch_X.shape[0] 
            outputs = model(batch_X, sbj_trials=current_sbj_trials)
            
            loss = criterion(outputs, batch_y)
            loss.backward()

            optimizer.step()
            
            epoch_loss += loss.item()

        if (epoch + 1) % 25 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # --- EVALUATION ON EXCLUDED SUBJECT ---
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask]
        
        # Pass the entire test subject in one go
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true_np = test_y.numpy()
        y_pred_np = predicted.cpu().numpy()

        all_y_true_latent.extend(y_true_np)
        all_y_pred_latent.extend(y_pred_np)

        # Standard accuracy is appropriate here as classes are balanced
        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results.append(acc)
        
    print(f"-> Finished Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

print("\n" + "="*50)
print(f"FINAL LOSO RESULTS (EEGNet + Latent Alignment)")
print(f"Mean Accuracy: {np.mean(loso_results)*100:.2f}% ± {np.std(loso_results)*100:.2f}%")
print("="*50)

all_y_true_latent = np.array(all_y_true_latent)
all_y_pred_latent = np.array(all_y_pred_latent)

mi_classes=['left_hand', 'right_hand']
calculate_advanced_metrics(all_y_true_latent, all_y_pred_latent, class_names=mi_classes)
plot_confusion_matrix(all_y_true_latent, all_y_pred_latent, class_names=mi_classes, title="EEGNet + Latent Alignment - Confusion Matrix")

Utilizzando il device: cpu

=== Inizio Validazione LOSO su PhysioNet MI (15 Soggetti) ===

-> Addestramento per Test Subject: 1
   Epoca [25/50], Loss: 0.5090
   Epoca [50/50], Loss: 0.3391
-> Fine Test Subject 1 | Accuracy: 66.67% | Tempo: 142.6s

-> Addestramento per Test Subject: 2
   Epoca [25/50], Loss: 0.4421
   Epoca [50/50], Loss: 0.3049
-> Fine Test Subject 2 | Accuracy: 75.56% | Tempo: 140.2s

-> Addestramento per Test Subject: 3
   Epoca [25/50], Loss: 0.5003
   Epoca [50/50], Loss: 0.2984
-> Fine Test Subject 3 | Accuracy: 46.67% | Tempo: 157.5s

-> Addestramento per Test Subject: 4
   Epoca [25/50], Loss: 0.4489
   Epoca [50/50], Loss: 0.2985
-> Fine Test Subject 4 | Accuracy: 62.22% | Tempo: 135.9s

-> Addestramento per Test Subject: 5
   Epoca [25/50], Loss: 0.4652
   Epoca [50/50], Loss: 0.3308
-> Fine Test Subject 5 | Accuracy: 40.00% | Tempo: 142.7s

-> Addestramento per Test Subject: 6
   Epoca [25/50], Loss: 0.4110
   Epoca [50/50], Loss: 0.2903
-> Fine Test Subject

```text
 Using device: cuda

=== Starting LOSO Validation on PhysioNet MI (15 Subjects) ===

-> Training for Test Subject: 1
   Epoch [25/50], Loss: 0.6754
   Epoch [50/50], Loss: 0.6417
-> Finished Test Subject 1 | Accuracy: 51.11% | Time: 9.8s

-> Training for Test Subject: 2
   Epoch [25/50], Loss: 0.6722
   Epoch [50/50], Loss: 0.6249
-> Finished Test Subject 2 | Accuracy: 62.22% | Time: 9.7s

-> Training for Test Subject: 3
   Epoch [25/50], Loss: 0.6764
   Epoch [50/50], Loss: 0.6434
-> Finished Test Subject 3 | Accuracy: 53.33% | Time: 9.9s

-> Training for Test Subject: 4
   Epoch [25/50], Loss: 0.6775
   Epoch [50/50], Loss: 0.6482
-> Finished Test Subject 4 | Accuracy: 55.56% | Time: 9.9s

-> Training for Test Subject: 5
   Epoch [25/50], Loss: 0.6725
   Epoch [50/50], Loss: 0.6438
-> Finished Test Subject 5 | Accuracy: 60.00% | Time: 9.9s

-> Training for Test Subject: 6
   Epoch [25/50], Loss: 0.6724
   Epoch [50/50], Loss: 0.6188
-> Finished Test Subject 6 | Accuracy: 48.89% | Time: 9.7s

-> Training for Test Subject: 7
   Epoch [25/50], Loss: 0.6703
   Epoch [50/50], Loss: 0.6299
-> Finished Test Subject 7 | Accuracy: 53.33% | Time: 9.7s

-> Training for Test Subject: 8
   Epoch [25/50], Loss: 0.6730
   Epoch [50/50], Loss: 0.6355
-> Finished Test Subject 8 | Accuracy: 44.44% | Time: 9.7s

-> Training for Test Subject: 9
   Epoch [25/50], Loss: 0.6745
   Epoch [50/50], Loss: 0.6527
-> Finished Test Subject 9 | Accuracy: 60.00% | Time: 9.7s

-> Training for Test Subject: 10
   Epoch [25/50], Loss: 0.6718
   Epoch [50/50], Loss: 0.6490
-> Finished Test Subject 10 | Accuracy: 46.67% | Time: 9.7s

-> Training for Test Subject: 11
   Epoch [25/50], Loss: 0.6733
   Epoch [50/50], Loss: 0.6450
-> Finished Test Subject 11 | Accuracy: 42.22% | Time: 9.7s

-> Training for Test Subject: 12
   Epoch [25/50], Loss: 0.6728
   Epoch [50/50], Loss: 0.6482
-> Finished Test Subject 12 | Accuracy: 57.78% | Time: 9.7s

-> Training for Test Subject: 13
   Epoch [25/50], Loss: 0.6713
   Epoch [50/50], Loss: 0.6268
-> Finished Test Subject 13 | Accuracy: 44.44% | Time: 9.7s

-> Training for Test Subject: 14
   Epoch [25/50], Loss: 0.6718
   Epoch [50/50], Loss: 0.6401
-> Finished Test Subject 14 | Accuracy: 60.00% | Time: 9.8s

-> Training for Test Subject: 15
   Epoch [25/50], Loss: 0.6777
   Epoch [50/50], Loss: 0.6438
-> Finished Test Subject 15 | Accuracy: 57.78% | Time: 9.8s

==================================================
FINAL LOSO RESULTS (EEGNet - Baseline)
Mean Accuracy: 53.19% ± 6.36%
==================================================

=======================================================
ADVANCED CLINICAL METRICS REPORT
=======================================================

COHEN'S KAPPA: 0.0606 (Slight/Poor Agreement)

SENSITIVITY (Recall Class 1): 40.54%
   (Ability to correctly identify target events)

SPECIFICITY (Recall Class 0): 65.50%
   (Ability to ignore background noise / avoid false alarms)

DETAILED REPORT (Scikit-Learn):
              precision    recall  f1-score   support

   left_hand       0.53      0.65      0.59       342
  right_hand       0.53      0.41      0.46       333

    accuracy                           0.53       675
   macro avg       0.53      0.53      0.52       675
weighted avg       0.53      0.53      0.52       675

=======================================================


![Confusion Matrix with Latent alignment](./confusion_matrix/confusion_matrix_eeg_net_latent.png)

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import accuracy_score
import time

# 1. Label Preparation (0 for left_hand, 1 for right_hand)
labels_dict = {val: i for i, val in enumerate(np.unique(y))}
y_int = np.array([labels_dict[label] for label in y], dtype=np.int64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y_int).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

# Training parameters
num_epochs = 50 # EEGNet is very lightweight, we can run more epochs
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
loso_results = []

subjects_array = metadata['subject'].unique()
print(f"\n=== Starting LOSO Validation on PhysioNet MI ({len(subjects_array)} Subjects) ===")

all_y_true_base = []
all_y_pred_base = []

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    # Masks for Training (N-1) and Test (1)
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    # Reinitialize the model from scratch for each LOSO iteration
    model = EEGNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='None').to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
# --- SPEED-UP TRAINING LOOP FOR BASELINE ---
    model.train()
    start_time = time.time()
    
    # Prepare unified training data (all subjects together)
    X_train_loso = X_tensor[train_mask].to(device)
    y_train_loso = y_tensor[train_mask].to(device)
    num_train_trials = X_train_loso.shape[0]

    for epoch in range(num_epochs):
        # Reset gradient once per epoch
        optimizer.zero_grad(set_to_none=True)
        
        # Pass the entire training block
        # In baseline mode, sbj_trials does not affect latent statistics
        outputs = model(X_train_loso, sbj_trials=num_train_trials)
        
        loss = criterion(outputs, y_train_loso)
        loss.backward()
        optimizer.step()

        # Log loss every 25 epochs
        if (epoch + 1) % 25 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")
            
    # --- EVALUATION ON EXCLUDED SUBJECT ---
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask]
        
        # Pass the entire test subject in one go
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)

        y_true_np = test_y.numpy()
        y_pred_np = predicted.cpu().numpy()

        all_y_true_base.extend(y_true_np)
        all_y_pred_base.extend(y_pred_np)
        
        # Standard accuracy is used as classes are balanced
        acc = accuracy_score(test_y.cpu().numpy(), predicted.cpu().numpy())
        loso_results.append(acc)
        
    print(f"-> Finished Test Subject {test_subject} | Accuracy: {acc*100:.2f}% | Time: {time.time()-start_time:.1f}s")

print("\n" + "="*50)
print(f"FINAL LOSO RESULTS (EEGNet - Baseline)")
print(f"Mean Accuracy: {np.mean(loso_results)*100:.2f}% ± {np.std(loso_results)*100:.2f}%")
print("="*50)

all_y_true_base = np.array(all_y_true_base)
all_y_pred_base = np.array(all_y_pred_base)

calculate_advanced_metrics(all_y_true_base, all_y_pred_base, class_names=mi_classes)
plot_confusion_matrix(all_y_true_base, all_y_pred_base, class_names=mi_classes, title="EEGNet - Baseline - Confusion Matrix")

Utilizzando il device: cpu

=== Inizio Validazione LOSO su PhysioNet MI (15 Soggetti) ===

-> Addestramento per Test Subject: 1
   Epoca [25/50], Loss: 0.6751
   Epoca [50/50], Loss: 0.6535
-> Fine Test Subject 1 | Accuracy: 57.78% | Tempo: 77.8s

-> Addestramento per Test Subject: 2
   Epoca [25/50], Loss: 0.6716
   Epoca [50/50], Loss: 0.6322
-> Fine Test Subject 2 | Accuracy: 60.00% | Tempo: 78.1s

-> Addestramento per Test Subject: 3
   Epoca [25/50], Loss: 0.6776
   Epoca [50/50], Loss: 0.6410
-> Fine Test Subject 3 | Accuracy: 57.78% | Tempo: 71.3s

-> Addestramento per Test Subject: 4
   Epoca [25/50], Loss: 0.6741
   Epoca [50/50], Loss: 0.6419
-> Fine Test Subject 4 | Accuracy: 57.78% | Tempo: 73.8s

-> Addestramento per Test Subject: 5
   Epoca [25/50], Loss: 0.6744
   Epoca [50/50], Loss: 0.6392
-> Fine Test Subject 5 | Accuracy: 60.00% | Tempo: 74.7s

-> Addestramento per Test Subject: 6
   Epoca [25/50], Loss: 0.6701
   Epoca [50/50], Loss: 0.6371
-> Fine Test Subject 6 | 

```text
 Using device: cuda

=== Starting LOSO Validation on PhysioNet MI (15 Subjects) ===

-> Training for Test Subject: 1
   Epoch [25/50], Loss: 0.6754
   Epoch [50/50], Loss: 0.6417
-> Finished Test Subject 1 | Accuracy: 51.11% | Time: 9.8s

-> Training for Test Subject: 2
   Epoch [25/50], Loss: 0.6722
   Epoch [50/50], Loss: 0.6249
-> Finished Test Subject 2 | Accuracy: 62.22% | Time: 9.7s

-> Training for Test Subject: 3
   Epoch [25/50], Loss: 0.6764
   Epoch [50/50], Loss: 0.6434
-> Finished Test Subject 3 | Accuracy: 53.33% | Time: 9.9s

-> Training for Test Subject: 4
   Epoch [25/50], Loss: 0.6775
   Epoch [50/50], Loss: 0.6482
-> Finished Test Subject 4 | Accuracy: 55.56% | Time: 9.9s

-> Training for Test Subject: 5
   Epoch [25/50], Loss: 0.6725
   Epoch [50/50], Loss: 0.6438
-> Finished Test Subject 5 | Accuracy: 60.00% | Time: 9.9s

-> Training for Test Subject: 6
   Epoch [25/50], Loss: 0.6724
   Epoch [50/50], Loss: 0.6188
-> Finished Test Subject 6 | Accuracy: 48.89% | Time: 9.7s

-> Training for Test Subject: 7
   Epoch [25/50], Loss: 0.6703
   Epoch [50/50], Loss: 0.6299
-> Finished Test Subject 7 | Accuracy: 53.33% | Time: 9.7s

-> Training for Test Subject: 8
   Epoch [25/50], Loss: 0.6730
   Epoch [50/50], Loss: 0.6355
-> Finished Test Subject 8 | Accuracy: 44.44% | Time: 9.7s

-> Training for Test Subject: 9
   Epoch [25/50], Loss: 0.6745
   Epoch [50/50], Loss: 0.6527
-> Finished Test Subject 9 | Accuracy: 60.00% | Time: 9.7s

-> Training for Test Subject: 10
   Epoch [25/50], Loss: 0.6718
   Epoch [50/50], Loss: 0.6490
-> Finished Test Subject 10 | Accuracy: 46.67% | Time: 9.7s

-> Training for Test Subject: 11
   Epoch [25/50], Loss: 0.6733
   Epoch [50/50], Loss: 0.6450
-> Finished Test Subject 11 | Accuracy: 42.22% | Time: 9.7s

-> Training for Test Subject: 12
   Epoch [25/50], Loss: 0.6728
   Epoch [50/50], Loss: 0.6482
-> Finished Test Subject 12 | Accuracy: 57.78% | Time: 9.7s

-> Training for Test Subject: 13
   Epoch [25/50], Loss: 0.6713
   Epoch [50/50], Loss: 0.6268
-> Finished Test Subject 13 | Accuracy: 44.44% | Time: 9.7s

-> Training for Test Subject: 14
   Epoch [25/50], Loss: 0.6718
   Epoch [50/50], Loss: 0.6401
-> Finished Test Subject 14 | Accuracy: 60.00% | Time: 9.8s

-> Training for Test Subject: 15
   Epoch [25/50], Loss: 0.6777
   Epoch [50/50], Loss: 0.6438
-> Finished Test Subject 15 | Accuracy: 57.78% | Time: 9.8s

==================================================
FINAL LOSO RESULTS (EEGNet - Baseline)
Mean Accuracy: 53.19% ± 6.36%
==================================================

=======================================================
ADVANCED CLINICAL METRICS REPORT
=======================================================

COHEN'S KAPPA: 0.0606 (Slight/Poor Agreement)

SENSITIVITY (Recall Class 1): 40.54%
   (Ability to correctly identify target events)

SPECIFICITY (Recall Class 0): 65.50%
   (Ability to ignore background noise / avoid false alarms)

DETAILED REPORT (Scikit-Learn):
              precision    recall  f1-score   support

   left_hand       0.53      0.65      0.59       342
  right_hand       0.53      0.41      0.46       333

    accuracy                           0.53       675
   macro avg       0.53      0.53      0.52       675
weighted avg       0.53      0.53      0.52       675

=======================================================


![Confusion Matrix with Latent alignment](./confusion_matrix/confusion_matrix_eeg_net_baseline.png)